# Tamil-English Code-Switched ASR: LoRA Fine-tuning

Fine-tunes **Whisper-small** with LoRA adapters on Tamil-English code-switched speech from IndicVoices.

**Runtime:** GPU (T4 or better) — set `Runtime > Change runtime type > T4 GPU`

**Required secrets** (set via `Secrets` panel in Colab, key icon in sidebar):
- `HF_TOKEN` — HuggingFace token with access to `ai4bharat/indicvoices`
- `WANDB_API_KEY` — Weights & Biases key (optional, skip if not tracking)

**Estimated time:** ~45 min for 1500 samples, 3 epochs on T4

## 1. Environment Setup

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — switch to GPU runtime')

In [ ]:
%%capture
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers>=4.40.0 peft>=0.10.0 datasets>=2.18.0
!pip install librosa soundfile jiwer evaluate langdetect
!pip install bitsandbytes accelerate scikit-learn pyyaml wandb python-dotenv
print('Installation complete')

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/indic_codeswitched_asr.git'  # update before running

if not os.path.exists('indic_codeswitched_asr'):
    !git clone {REPO_URL}

%cd indic_codeswitched_asr
print('Working directory:', os.getcwd())

In [ ]:
# Load credentials from Colab Secrets
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    wandb_key = userdata.get('WANDB_API_KEY')
except Exception:
    # Fallback: paste tokens directly (do not commit)
    hf_token = ''   # paste here if not using Secrets
    wandb_key = ''  # optional

# Write .env so existing scripts pick it up
with open('.env', 'w') as f:
    f.write(f'HF_TOKEN={hf_token}\n')
    if wandb_key:
        f.write(f'WANDB_API_KEY={wandb_key}\n')

print('Credentials set' if hf_token else 'WARNING: HF_TOKEN is empty')

## 2. Data Preparation

Streams from `ai4bharat/indicvoices` (Tamil), resamples to 16kHz, tags each segment as
`monolingual_tamil`, `monolingual_english`, or `code_switched`, and builds a stratified 80/10/10 split.

In [ ]:
MAX_SAMPLES = 1500  # reduce to 300 for a quick smoke-test

import sys
sys.path.insert(0, '.')

from data.prepare_dataset import (
    authenticate_hf, load_indicvoices_tamil,
    build_dataset_splits, print_dataset_stats
)

authenticate_hf()
samples = load_indicvoices_tamil(max_samples=MAX_SAMPLES)
print_dataset_stats(samples)

In [ ]:
splits = build_dataset_splits(samples)
print(f"Train: {len(splits['train'])}  Val: {len(splits['validation'])}  Test: {len(splits['test'])}")

# Verify code-switched proportion in training split
from collections import Counter
train_types = Counter(s['segment_type'] for s in splits['train'])
print('Segment type distribution (train):', dict(train_types))

## 3. Fine-tuning

- **Model:** Whisper-small
- **LoRA:** r=32, alpha=64, targets `q_proj` + `v_proj`
- **Sampling:** code-switched ×3, high-switch ×2, monolingual ×0.5
- **Optimizer:** AdamW 8-bit, FP16

In [ ]:
import yaml
with open('fine_tuning/config.yaml') as f:
    config = yaml.safe_load(f)

print('=== Training configuration ===')
print(yaml.dump(config, default_flow_style=False))

In [ ]:
from fine_tuning.train import (
    load_config, load_model_and_processor, apply_lora,
    oversample_by_type, prepare_dataset_for_training,
    DataCollatorSpeechSeq2SeqWithPadding, compute_metrics_fn
)
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback
import os, torch, wandb
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
WANDB_KEY = os.getenv('WANDB_API_KEY')

config = load_config()

if WANDB_KEY:
    wandb.login(key=WANDB_KEY)
    wandb.init(
        project='indic-codeswitched-asr',
        name='whisper-small-lora-tanglish-colab',
        config=config
    )

processor, model = load_model_and_processor(config)
model = apply_lora(model, config)

train_samples = oversample_by_type(splits['train'], config)
print(f'Training samples after oversampling: {len(train_samples)}')

train_data = prepare_dataset_for_training(train_samples, processor)
val_data   = prepare_dataset_for_training(splits['validation'], processor)
print(f'Prepared {len(train_data)} train, {len(val_data)} val features')

In [ ]:
training_cfg = config['training']

training_args = Seq2SeqTrainingArguments(
    output_dir=training_cfg['output_dir'],
    num_train_epochs=training_cfg['num_train_epochs'],
    per_device_train_batch_size=training_cfg['per_device_train_batch_size'],
    gradient_accumulation_steps=training_cfg['gradient_accumulation_steps'],
    learning_rate=training_cfg['learning_rate'],
    warmup_steps=training_cfg['warmup_steps'],
    eval_strategy=training_cfg['evaluation_strategy'],
    eval_steps=training_cfg['eval_steps'],
    save_steps=training_cfg['save_steps'],
    logging_steps=training_cfg['logging_steps'],
    fp16=training_cfg['fp16'] and DEVICE == 'cuda',
    dataloader_num_workers=training_cfg['dataloader_num_workers'],
    load_best_model_at_end=training_cfg['load_best_model_at_end'],
    metric_for_best_model=training_cfg['metric_for_best_model'],
    greater_is_better=training_cfg['greater_is_better'],
    optim=training_cfg['optim'],
    predict_with_generate=True,
    generation_max_length=256,
    report_to='wandb' if WANDB_KEY else 'none',
    push_to_hub=False,
)

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics_fn(processor),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

In [ ]:
best_model_path = Path(training_cfg['output_dir']) / 'best_model'
trainer.save_model(str(best_model_path))
processor.save_pretrained(str(best_model_path))
print(f'Model saved to {best_model_path}')

if wandb.run:
    wandb.finish()

## 4. Evaluation

Run WER evaluation on the held-out test set, stratified by segment type.

In [ ]:
from evaluation.baseline_eval import evaluate_model

finetuned_config = {
    'type': 'whisper',
    'name': str(best_model_path),
    'language': 'ta',
    'task': 'transcribe'
}

results = evaluate_model(
    model_key='whisper_small_lora',
    model_config=finetuned_config,
    test_samples=splits['test'],
    max_samples=None  # full test set
)

print('\n=== Fine-tuned Model Results ===')
print(f"Overall WER:         {results['overall_wer']}")
print(f"Monolingual Tamil:   {results['monolingual_tamil_wer']}")
print(f"Monolingual English: {results['monolingual_english_wer']}")
print(f"Code-switched WER:   {results['code_switched_wer']}")

## 5. Push to HuggingFace Hub (optional)

In [ ]:
HF_REPO_ID = 'YOUR_USERNAME/whisper-small-tanglish-lora'  # update before running

model.push_to_hub(HF_REPO_ID, token=os.getenv('HF_TOKEN'))
processor.push_to_hub(HF_REPO_ID, token=os.getenv('HF_TOKEN'))
print(f'Model pushed to https://huggingface.co/{HF_REPO_ID}')

## 6. Failure Analysis Report

In [ ]:
import json
from pathlib import Path

# Save fine-tuned results alongside baseline results for comparison
results_dir = Path('results')
results_dir.mkdir(exist_ok=True)

with open(results_dir / 'whisper_small_lora_wer.json', 'w') as f:
    json.dump(results, f, indent=2)

# If baseline results exist, build a combined comparison
combined_path = results_dir / 'baseline_wer_all.json'
if combined_path.exists():
    with open(combined_path) as f:
        all_results = json.load(f)
    all_results['whisper_small_lora'] = results
    with open(combined_path, 'w') as f:
        json.dump(all_results, f, indent=2)

    from analysis.report import build_markdown, build_summary
    md = build_markdown(all_results)
    print(md[:3000], '\n...(truncated)')
else:
    print('No baseline_wer_all.json found — run evaluation/baseline_eval.py first to compare models.')